# RetailIQ - Data Understanding and Sales Trends (ML-13)

This notebook inspects the finalized leakage-safe processed dataset and explores historical sales trends. It does not preprocess data, modify the source CSV, or train models.

**Dataset:** `data/processed/retailiq_leakage_safe.csv`  
**Feature list:** `data/reference/feature_columns.txt`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("../data/processed/retailiq_leakage_safe.csv")
FEATURE_PATH = Path("../data/reference/feature_columns.txt")
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
feature_columns = [line.strip() for line in FEATURE_PATH.read_text().splitlines() if line.strip()]
print(f"Loaded {DATA_PATH} | {len(df):,} rows x {len(df.columns):,} columns")
print(f"Feature list: {len(feature_columns)} entries")

## 1. Dataset verification

The following checks describe the provided dataset as-is. Missing values in lag features and transactions are reported rather than imputed.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes.to_string())
print("\nMissing values by column:\n", df.isna().sum().to_string())
print("\nDuplicate rows:", int(df.duplicated().sum()))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Number of stores:", df["store_nbr"].nunique())
print("Number of product families:", df["family"].nunique())
print("\nSales descriptive statistics:\n", df["sales"].describe().to_string())
print("Zero sales rows:", int(df["sales"].eq(0).sum()))
print("Negative sales rows:", int(df["sales"].lt(0).sum()))
print("\nFeature columns listed in feature_columns.txt:\n", feature_columns)

## 2. Sales trends

Sales are aggregated across stores and product families for overall time trends.

In [ ]:
daily_sales = df.groupby("date", sort=True)["sales"].sum().rename("sales")
fig, ax = plt.subplots(figsize=(14, 5))
daily_sales.plot(ax=ax, linewidth=0.8, color="steelblue")
ax.set(title="Daily Total Sales", xlabel="Date", ylabel="Total sales")
fig.tight_layout()
plt.show()

In [ ]:
monthly_sales = daily_sales.resample("MS").sum()
fig, ax = plt.subplots(figsize=(14, 5))
monthly_sales.plot(ax=ax, marker="o", markersize=3, linewidth=1.2, color="darkorange")
ax.set(title="Monthly Total Sales", xlabel="Month", ylabel="Total sales")
fig.tight_layout()
plt.show()

In [ ]:
weekly_sales = daily_sales.resample("W").sum()
fig, ax = plt.subplots(figsize=(14, 5))
weekly_sales.plot(ax=ax, linewidth=1, color="seagreen")
ax.set(title="Weekly Total Sales", xlabel="Week ending", ylabel="Total sales")
fig.tight_layout()
plt.show()

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sales_by_weekday = (df.assign(day_name=df["date"].dt.day_name())
                     .groupby("day_name")["sales"].mean()
                     .reindex(weekday_order))
fig, ax = plt.subplots(figsize=(10, 4))
sales_by_weekday.plot(kind="bar", ax=ax, color="slateblue")
ax.set(title="Average Sales by Day of Week", xlabel="Day of week", ylabel="Average sales per store-family row")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()
sales_by_weekday

In [ ]:
year_month_sales = df.groupby(["year", "month"], sort=True)["sales"].sum().unstack("month")
fig, ax = plt.subplots(figsize=(12, 5))
image = ax.imshow(year_month_sales, aspect="auto", cmap="YlGnBu")
ax.set(title="Total Sales by Year and Month", xlabel="Month", ylabel="Year",
       xticks=range(12), xticklabels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
       yticks=range(len(year_month_sales.index)), yticklabels=year_month_sales.index)
fig.colorbar(image, ax=ax, label="Total sales")
fig.tight_layout()
plt.show()

## 3. Promotion and transaction relationships

These summaries describe associations in the processed data, not causal estimates. `onpromotion` is the number of promoted items for a store-date-family row; the comparison uses row-level sales for rows with versus without a positive promotion count.

In [ ]:
promotion_summary = (df.assign(promotion_status=df["onpromotion"].gt(0).map({True: "Promotion count > 0", False: "Promotion count = 0"}))
                     .groupby("promotion_status")["sales"]
                     .agg(rows="size", total_sales="sum", mean_sales_per_row="mean", median_sales_per_row="median"))
display(promotion_summary)
fig, ax = plt.subplots(figsize=(8, 4))
promotion_summary["mean_sales_per_row"].plot(kind="bar", ax=ax, color=["gray", "darkorange"])
ax.set(title="Average Row Sales by Promotion Status", xlabel="Promotion status", ylabel="Average sales per row")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
# Transactions repeat across product-family rows for a store-date. Aggregate sales
# to store-date and keep one transaction value per store-date to avoid duplication.
store_day = (df.groupby(["date", "store_nbr"], as_index=False)
             .agg(sales=("sales", "sum"), transactions=("transactions", "first")))
valid_transactions = store_day.dropna(subset=["transactions"])
print(f"Store-days with transactions available: {len(valid_transactions):,} / {len(store_day):,}")
print("Store-day sales / transaction correlation:",
      valid_transactions["sales"].corr(valid_transactions["transactions"]))
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(valid_transactions["transactions"], valid_transactions["sales"], alpha=0.18, s=10, color="teal")
ax.set(title="Store-day Sales vs Transactions", xlabel="Transactions", ylabel="Total sales")
fig.tight_layout()
plt.show()

## Interpretation notes

Sales totals aggregate all store-family rows for each date. The weekday chart reports mean sales per store-family row. Transaction analysis aggregates sales to store-day level because transaction counts repeat across product families. Promotions, sales, and transactions may be associated with store, family, date, and other factors; these plots alone do not establish causality.